# Model Creation
## Introduction
This notebook covers the final formatting steps required to match the __[PyPSA](https://pypsa.readthedocs.io/en/latest/)__ input format.

In [1]:
import pandas as pd
import geopandas as gpd
import os
from shapely import Point
import pypsa
import numpy as np
from helper_functions import mapPoints

In [2]:
### LOADING DATA
# Setting paths
path = os.getcwd()
results_path = os.path.join(path, 'results')
CODERS_path = os.path.join(path, 'data', 'CODERS')
model_results_path = os.path.join(path, 'model', 'national_model')
os.makedirs(model_results_path, exist_ok=True)

In [3]:
LOAD_SCENARIO = 'CNZ'
REF_YEAR = 2025
BUILD_YEAR = 2030 # Earliest year new projects can come online

In [6]:
# Dicts
gen_type_map = {
    'biomass': 'default_biomass',
    'Biomass': 'default_biomass',
    'MSW': 'default_biomass',
    'NG_CC': 'gas_CC',
    'NG_CG': 'gas_CC', # Need to change
    'NG_SC': 'gas_CT', # Need to confirm
    'biogas': 'default_biogas',
    'coal': 'coal_IGCC', # Need to confirm/change
    'coal_CCS': 'coal_IGCC', # Need to add new type
    'diesel_CT': 'default_diesel',
    'gasoline_CT': 'default_diesel', # Need to add new type?
    'hydro_daily': 'hydro_storage',
    'hydro_monthly': 'hydro_storage',
    'hydro_run': 'hydro',
    'nuclear': 'default_nuclear',
    'oil_CT': 'default_oil', # Need to add new type?
    'oil_ST': 'default_oil',
    'solar_PV': 'default_solar_PV',
    'wind_ons': 'wind_2021'
}

# Reading cluster data
clusters = gpd.read_feather(os.path.join(path, 'results', 'clustered_zone_data.feather')).to_crs('EPSG: 3347')
clusters = clusters[['geometry']]

c:\Users\ndematos\envs\pypsa_canada_py312\Lib\site-packages\geopandas\io\arrow.py:878: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  table = feather.read_table(path, columns=columns, **kwargs)


## Buses
This step creates the buses input file.

In [7]:
buses = pd.read_csv(os.path.join(results_path, 'nearest_nodes.csv'), index_col=0)
buses = buses.drop('node', axis=1).rename_axis('name').rename(columns={'lat':'y', 'lon':'x'})
buses['province'] = buses.index.str[:2]
buses.to_csv(os.path.join(model_results_path, 'buses.csv'))

## Generators
This step combines the hydro, wind/solar and combustion generators into a single file. The time series capacity factor data for wind/solar and hydro are also combined.
### Combustion generators
For combustion generators, the CODERS database is used. The previous renewal year field in the generators table is assumed to be the "start year" for the purposes of calculating the retirement year. The lifetimes are added in the generator pre-processing step in the pypsa_cad model. The units are dropped if they are assumed to retire before 2025, based on the renewal year and assumed lifetime.

In [8]:
# Aggregate the generators based on bus, model, retirement year (rounded) and max_hours
aggregate_gens = True

# Generator Lifetimes
gen_life = {
    'default_biomass': 30,
    'default_biogas': 30,
    'coal_IGCC': 50,
    'coal_pulverized': 50,
    'default_diesel': 40,
    'gas_CC': 40,
    'gas_CT': 40,
    'default_nuclear': 80, # Over-estimated, manually set some of the retirement years below
    'default_SMR': 30,
    'default_oil': 40 
}

In [ ]:
def find_cluster(df):
    # Find nearest cluster
    df = mapPoints(df)
    df = df.to_crs('EPSG:3347')
    df = df.sjoin_nearest(clusters, how='left')
    df = df.drop('geometry', axis=1)
    df = df.rename({'cluster':'bus'},axis=1)
    return df

In [ ]:
# Existing generator data
generator_data = pd.read_csv(os.path.join(CODERS_path, 'generators.csv'))
combustion_gens_data = generator_data[~generator_data.gen_type.isin(['hydro_run', 'hydro_daily', 'hydro_monthly', 'solar_PV', 'wind_ons'])]

# If the renewal year is greater than reference year but the start year is before reference year, set the renewal to reference year
combustion_gens_data.loc[(combustion_gens_data.previous_renewal_year >= REF_YEAR) & (combustion_gens_data.start_year < REF_YEAR), 'previous_renewal_year'] = REF_YEAR

# Formatting
combustion_gens_data = combustion_gens_data[['generation_unit_name', 'unit_installed_capacity', 'previous_renewal_year', 'gen_type', 'latitude', 'longitude', 'province', 'closure_year']]
combustion_gens_data = combustion_gens_data.rename({'generation_unit_name':'name', 'unit_installed_capacity':'p_nom', 'gen_type':'model', 'previous_renewal_year': 'build_year', 'closure_year': 'retirement_year'}, axis=1)

# Planned generator data
planned_gens = pd.read_csv(os.path.join(path, 'data', 'planned_projects', 'new_capacity.csv'))
planned_gens = planned_gens[~planned_gens.model.isin(['hydro_run', 'hydro_daily', 'hydro_monthly', 'solar_PV', 'wind_ons'])]
combustion_gens_data = pd.concat([combustion_gens_data, planned_gens])

# Generator Models and assumed lifetimes
combustion_gens_data.loc[:, 'model'] = combustion_gens_data.loc[:, 'model'].replace(gen_type_map)
combustion_gens_data['lifetime'] = combustion_gens_data['model'].map(gen_life)

# Load manual retirements
retirements = pd.read_csv(os.path.join(path, 'data', 'planned_projects', 'thermal_retirements.csv'))
retirements = retirements[~retirements.CODERS_name.isna()]
retirements = retirements.set_index('CODERS_name')['year']
retirements = retirements.to_dict()

## Calculate retirement years
# Replace assumed values with retirements specified in thermal_retirements.csv
combustion_gens_data['retirement_year'] = combustion_gens_data['retirement_year'].fillna(combustion_gens_data['name'].map(retirements))
# Fill missing values with assumed lifetime
combustion_gens_data['retirement_year'] = combustion_gens_data['retirement_year'].fillna(combustion_gens_data['build_year'] + combustion_gens_data['lifetime'])
# Round up to nearest 5th year
combustion_gens_data['retirement_year'] = (5 * np.ceil((combustion_gens_data['retirement_year']) / 5)).astype(int)
# Remove generators which retire before model start
combustion_gens_data = combustion_gens_data[combustion_gens_data['retirement_year'] > REF_YEAR]

# Start year doesn't matter as long as its less than base year
combustion_gens_data.loc[(combustion_gens_data.build_year < REF_YEAR), 'start_year'] = int(REF_YEAR)
# Retirement year doesn't matter if its greater than 2050
combustion_gens_data.loc[(combustion_gens_data.retirement_year > 2050), 'retirement_year'] = 2055

# Map to clusters
combustion_gens_data = find_cluster(combustion_gens_data)

if aggregate_gens:
    combustion_gens_data = combustion_gens_data.groupby(['cluster', 'model', 'start_year', 'retirement_year']).sum()['p_nom'].reset_index()
    combustion_gens_data['name'] = combustion_gens_data.cluster + '_' + combustion_gens_data.model + '_' + combustion_gens_data.start_year.astype(str) + '_to_' + combustion_gens_data.retirement_year.astype(str)
else:
    combustion_gens_data = combustion_gens_data.groupby(['generation_unit_name', 'cluster', 'model', 'start_year', 'retirement_year']).sum()['p_nom'].reset_index()
    combustion_gens_data['name'] = combustion_gens_data.generation_unit_name + '_' + combustion_gens_data.model + '_' + combustion_gens_data.start_year.astype(str) + '_to_' + combustion_gens_data.retirement_year.astype(str)

# Format generators
combustion_gens_data['p_nom_extendable'] = False
combustion_gens_data['lifetime'] = combustion_gens_data['retirement_year'].subtract(combustion_gens_data['build_year'])
combustion_gens_data = combustion_gens_data.set_index('name',drop=True)
combustion_gens_data = combustion_gens_data[['bus', 'model', 'p_nom', 'p_nom_extendable', 'build_year', 'lifetime']]
combustion_gens_data.head()

,name,p_nom,build_year,model,latitude,longitude,province,retirement_year,lifetime
193,Base Plant,50.0,1967,gas_CC,57.005646,-111.479236,AB,2010,40
200,Sundance Coal_04,406.0,2007,coal_IGCC,53.507446,-114.556785,AB,2020,50
201,Sundance Coal_06,401.0,2001,coal_IGCC,53.507446,-114.556785,AB,2020,50
209,Sheerness Coal_02,408.0,1990,coal_IGCC,51.442549,-111.791788,AB,2020,50
302,Catalyst Paper Crofton,38.0,1990,default_biomass,48.875707,-123.650029,BC,2010,30
431,Catalyst Paper Port Alberni,17.0,1990,default_biomass,49.247371,-124.811185,BC,2010,30
1248,Summerside_01,2.2,1965,default_diesel,46.393612,-63.789311,PE,2005,40
1249,Summerside_02,2.0,1965,default_diesel,46.393612,-63.789311,PE,2005,40
1250,Summerside_03,2.0,1965,default_diesel,46.393612,-63.789311,PE,2005,40
1251,Summerside_05,2.2,1965,default_diesel,46.393612,-63.789311,PE,2005,40


In [ ]:


combustion_gens['start_year'] = (5 * np.ceil((combustion_gens['previous_renewal_year']) / 5)).astype(int)

# First find closure years in CODERS
combustion_gens['retirement_year'] = combustion_gens['closure_year']
# Fill missing values with assumed lifetime
combustion_gens['retirement_year'] = combustion_gens['retirement_year'].fillna(combustion_gens['start_year'] + combustion_gens['lifetime'])
# Manually set some closure years from planned retirements
combustion_gens['retirement_year'] = combustion_gens['generation_unit_name'].map(retirements).fillna(combustion_gens.retirement_year)
# Round to nearest 5th year
combustion_gens['retirement_year'] = (5 * np.ceil((combustion_gens['retirement_year']) / 5)).astype(int)

combustion_gens = combustion_gens[combustion_gens['retirement_year'] > REF_YEAR] # Remove generators which retire before model start

# Start year doesn't matter as long as its less than base year
combustion_gens.loc[(combustion_gens.start_year < REF_YEAR), 'start_year'] = int(REF_YEAR)
# Retirement year doesn't matter if its greater than 2050
combustion_gens.loc[(combustion_gens.retirement_year > 2050), 'retirement_year'] = 2055

In [ ]:
def find_cluster(df):
    # Find nearest cluster
    df = mapPoints(df)
    df = df.to_crs('EPSG:3347')
    df = df.sjoin_nearest(clusters, how='left')
    df = df.drop('geometry', axis=1)
    return df

In [9]:
generator_data = pd.read_csv(os.path.join(CODERS_path, 'generators.csv'))
combustion_gens_data = generator_data[~generator_data.gen_type.isin(['hydro_run', 'hydro_daily', 'hydro_monthly', 'solar_PV', 'wind_ons'])]

combustion_gens_data = mapPoints(combustion_gens_data)
combustion_gens_data = combustion_gens_data.to_crs('EPSG:3347')

# Finding cluster
combustion_gens_data = combustion_gens_data.sjoin_nearest(clusters, how='left')
combustion_gens_data = combustion_gens_data.drop('geometry', axis=1)

c:\Users\ndematos\envs\pypsa_canada_py312\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [10]:
retirements = pd.read_csv(os.path.join(path, 'data', 'planned_projects', 'thermal_retirements.csv'))
retirements = retirements[~retirements.CODERS_name.isna()]
retirements = retirements.set_index('CODERS_name')['year']
retirements = retirements.to_dict()

In [60]:
combustion_gens = combustion_gens_data.copy()

# Map generation types
combustion_gens.loc[:, 'model'] = combustion_gens.loc[:, 'gen_type'].map(gen_type_map)

# lifetimes
combustion_gens['lifetime'] = combustion_gens['model'].map(gen_life)

# If the renewal year is greater than reference year but the start year is before reference year, set the renewal to reference year
combustion_gens_data.loc[(combustion_gens_data.previous_renewal_year >= REF_YEAR) & (combustion_gens_data.start_year < REF_YEAR), 'previous_renewal_year'] = REF_YEAR
combustion_gens = combustion_gens[(combustion_gens.previous_renewal_year + combustion_gens.lifetime) >= REF_YEAR]

combustion_gens['start_year'] = (5 * np.ceil((combustion_gens['previous_renewal_year']) / 5)).astype(int)

# First find closure years in CODERS
combustion_gens['retirement_year'] = combustion_gens['closure_year']
# Fill missing values with assumed lifetime
combustion_gens['retirement_year'] = combustion_gens['retirement_year'].fillna(combustion_gens['start_year'] + combustion_gens['lifetime'])
# Manually set some closure years from planned retirements
combustion_gens['retirement_year'] = combustion_gens['generation_unit_name'].map(retirements).fillna(combustion_gens.retirement_year)
# Round to nearest 5th year
combustion_gens['retirement_year'] = (5 * np.ceil((combustion_gens['retirement_year']) / 5)).astype(int)

combustion_gens = combustion_gens[combustion_gens['retirement_year'] > REF_YEAR] # Remove generators which retire before model start

# Start year doesn't matter as long as its less than base year
combustion_gens.loc[(combustion_gens.start_year < REF_YEAR), 'start_year'] = int(REF_YEAR)
# Retirement year doesn't matter if its greater than 2050
combustion_gens.loc[(combustion_gens.retirement_year > 2050), 'retirement_year'] = 2055

combustion_gens['p_nom'] = combustion_gens['unit_installed_capacity']

if aggregate_gens:
    combustion_gens = combustion_gens.groupby(['cluster', 'model', 'start_year', 'retirement_year']).sum()['p_nom'].reset_index()
    combustion_gens['name'] = combustion_gens.cluster + '_' + combustion_gens.model + '_' + combustion_gens.start_year.astype(str) + '_to_' + combustion_gens.retirement_year.astype(str)
else:
    combustion_gens = combustion_gens.groupby(['generation_unit_name', 'cluster', 'model', 'start_year', 'retirement_year']).sum()['p_nom'].reset_index()
    combustion_gens['name'] = combustion_gens.generation_unit_name + '_' + combustion_gens.model + '_' + combustion_gens.start_year.astype(str) + '_to_' + combustion_gens.retirement_year.astype(str)

# Non-extendable for existing generators
combustion_gens['p_nom_extendable'] = False

# Format generators
combustion_gens['lifetime'] = combustion_gens['retirement_year'].subtract(combustion_gens['start_year'])
combustion_gens = combustion_gens.rename(columns={'cluster':'bus', 'start_year':'build_year'})
combustion_gens = combustion_gens.set_index('name',drop=True)
combustion_gens = combustion_gens[['bus', 'model', 'p_nom', 'p_nom_extendable', 'build_year', 'lifetime']]

combustion_gens.head()

,bus,model,p_nom,p_nom_extendable,build_year,lifetime
name,,,,,,
AB_Central_gas_CC_2025_to_2035,AB_Central,gas_CC,1191.00,False,2025,10
AB_Central_gas_CC_2025_to_2040,AB_Central,gas_CC,2809.00,False,2025,15
AB_Central_gas_CC_2025_to_2045,AB_Central,gas_CC,287.00,False,2025,20
AB_Central_gas_CC_2025_to_2050,AB_Central,gas_CC,19.00,False,2025,25
AB_Central_gas_CC_2025_to_2055,AB_Central,gas_CC,161.05,False,2025,30


#### Planned Generators

In [61]:
# Planned generator data
planned_gens = pd.read_csv(os.path.join(path, 'data', 'planned_projects', 'new_capacity.csv'))
planned_gens = planned_gens[~planned_gens.model.isin(['hydro_run', 'hydro_daily', 'hydro_monthly', 'solar_PV', 'wind_ons'])]
planned_gens['geometry'] = planned_gens.apply(lambda row: Point(row.longitude, row.latitude), axis=1)
planned_gens = gpd.GeoDataFrame(planned_gens, geometry=planned_gens.geometry, index=planned_gens.index, crs='EPSG:4326').to_crs('EPSG: 3347')
planned_gens = planned_gens.sjoin_nearest(clusters, how='left')
planned_gens = planned_gens.drop(['geometry', 'province'], axis=1)
planned_gens = planned_gens.rename({'cluster':'bus'}, axis=1)
planned_gens['p_nom_extendable'] = False
planned_gens['lifetime'] = planned_gens['model'].map(gen_life)

planned_gens['build_year'] = (5 * np.ceil((planned_gens['build_year']) / 5)).astype(int)
planned_gens['retirement_year'] = (5 * np.ceil((planned_gens['build_year'] + planned_gens['lifetime']) / 5)).astype(int)
planned_gens['retirement_year'] = planned_gens['name'].map(retirements).fillna(planned_gens.retirement_year)
planned_gens['retirement_year'] = (5 * np.ceil((planned_gens['retirement_year']) / 5)).astype(int)

planned_gens['lifetime'] = planned_gens['retirement_year'].subtract(planned_gens['build_year'])

planned_gens['name'] = planned_gens.name + '_' + planned_gens.model + '_' + planned_gens.build_year.astype(str) + '_to_' + planned_gens.retirement_year.astype(str)

planned_gens = planned_gens.set_index('name', drop=True)
planned_gens = planned_gens.drop(['latitude', 'longitude', 'retirement_year'], axis=1)

combustion_gens = pd.concat([combustion_gens, planned_gens])
combustion_gens.tail()

,bus,model,p_nom,p_nom_extendable,build_year,lifetime
name,,,,,,
Shand Life Extension_coal_pulverized_2035_to_2085,SK,coal_pulverized,276.0,False,2035,50
Poplar River 1 Life Extension_coal_pulverized_2035_to_2085,SK,coal_pulverized,291.0,False,2035,50
Poplar River 2 Life Extension_coal_pulverized_2035_to_2085,SK,coal_pulverized,291.0,False,2035,50
Brandon Additional Turbines_gas_CT_2030_to_2070,MB,gas_CT,750.0,False,2030,40
Avalon CT_gas_CT_2030_to_2070,NL_Newfoundland,gas_CT,150.0,False,2030,40


#### Optional Generators
Adding expandable natural gas generators at each bus

In [62]:
# Skips locations where gas infrastructure is lacking
no_gas = ['ON_Toronto', 'PE', 'QC_North', 'QC_Central', 'NL_Labrador', 'NL_Newfoundland'] 
optional_gens = {}
for bus in buses.index:
    if bus in no_gas:
        continue
    # Add natural gas generator
    optional_gens[f'{bus}_OPT_gas'] = {
        'bus':bus,
        'model':'gas_CC',
        'group':'ungrouped',
        'p_nom':0,
        'p_nom_extendable':True,
        'build_year':BUILD_YEAR
    }
optional_gens = pd.DataFrame.from_dict(optional_gens).T
optional_gens

,bus,model,group,p_nom,p_nom_extendable,build_year
AB_Central_OPT_gas,AB_Central,gas_CC,ungrouped,0,True,2030
AB_North_OPT_gas,AB_North,gas_CC,ungrouped,0,True,2030
AB_South_OPT_gas,AB_South,gas_CC,ungrouped,0,True,2030
BC_North_OPT_gas,BC_North,gas_CC,ungrouped,0,True,2030
BC_South_OPT_gas,BC_South,gas_CC,ungrouped,0,True,2030
MB_OPT_gas,MB,gas_CC,ungrouped,0,True,2030
NB_East_OPT_gas,NB_East,gas_CC,ungrouped,0,True,2030
NB_West_OPT_gas,NB_West,gas_CC,ungrouped,0,True,2030
NS_East_OPT_gas,NS_East,gas_CC,ungrouped,0,True,2030
NS_West_OPT_gas,NS_West,gas_CC,ungrouped,0,True,2030


#### Wind, solar and hydro
Appending the wind, solar and hydro generators to the generator file, as well as saving the hourly capacity factors.

In [63]:
# Generators
generators = pd.concat([
    combustion_gens,
    optional_gens,
    pd.read_csv(os.path.join(results_path, 'wind_solar_gens.csv'), index_col=0), 
    pd.read_csv(os.path.join(results_path, 'ror_gens.csv'), index_col=0)])
generators.index.name = 'name'
generators.to_csv(os.path.join(model_results_path, 'generators.csv'))

# Capacity Factors
cf_data = pd.concat([
    pd.read_csv(os.path.join(results_path, 'wind_solar_cf.csv'), index_col=0).reset_index(drop=True),
    pd.read_csv(os.path.join(results_path, 'ror_cf.csv'), index_col=0).reset_index(drop=True)], 
    axis=1)
cf_data.index = pd.date_range(f'{REF_YEAR}-01-01', periods=8760, freq='h')
cf_data.to_csv(os.path.join(model_results_path, 'generators-p_max_pu.csv'))

## Storage Units
### Existing storage

In [ ]:
# Aggregate the storage based on bus, model, retirement year (rounded) and max_hours
aggregate_storage = True

storage_data = pd.read_csv(os.path.join(CODERS_path, 'storage.csv'))
# Only lithium batteries for now
existing_storage_units = storage_data[storage_data.storage_type == 'storage_lithium']
# Function to create geopandas dataframe of points at each node
existing_storage_units = mapPoints(existing_storage_units)
existing_storage_units = existing_storage_units.to_crs('EPSG: 3347')

# Finding cluster
existing_storage_units = existing_storage_units.sjoin_nearest(clusters, how='left')
existing_storage_units = existing_storage_units.drop('geometry',axis=1)

# Map generation types
existing_storage_units.loc[:, 'model'] = 'default_liion_battery'

# If the renewal year is greater than REF_YEAR but the start year is before REF_YEAR, set the renewal to REF_YEAR
existing_storage_units.loc[(existing_storage_units.previous_renewal_year >= REF_YEAR) & (existing_storage_units.start_year < REF_YEAR), 'previous_renewal_year'] = REF_YEAR
# Using the renewal year as the start year, unless the start year is greater than REF_YEAR (planned capacity)
existing_storage_units.loc[existing_storage_units.start_year > REF_YEAR, 'previous_renewal_year'] = existing_storage_units.loc[existing_storage_units.start_year > REF_YEAR, 'start_year']
existing_storage_units['build_year'] = (5 * np.ceil((existing_storage_units['previous_renewal_year']) / 5)).astype(int)

# Lifetimes
existing_storage_units['lifetime'] = 25
existing_storage_units['retirement_year'] = (5 * np.ceil((existing_storage_units['build_year'] + existing_storage_units['lifetime']) / 5)).astype(int)
existing_storage_units = existing_storage_units.rename({'storage_duration':'max_hours', 'storage_capacity': 'p_nom', 'storage_facility_name': 'name', 'cluster':'bus'}, axis=1)

if aggregate_storage:
    existing_storage_units = existing_storage_units.groupby(['bus', 'model', 'retirement_year', 'build_year', 'max_hours']).sum()['p_nom'].reset_index()
    existing_storage_units['name'] = existing_storage_units.bus + '_' + existing_storage_units.model + '_' + existing_storage_units.max_hours.astype(str) + 'h_' + existing_storage_units.build_year.astype(str) + '_to_' + existing_storage_units.retirement_year.astype(str)

else:
    existing_storage_units = existing_storage_units.groupby(['name', 'bus', 'model', 'build_year', 'retirement_year', 'max_hours']).sum()['p_nom'].reset_index()
    existing_storage_units['name'] = existing_storage_units.name + '_' + existing_storage_units.model + '_' + existing_storage_units.max_hours.astype(str) + 'h_' + existing_storage_units.build_year.astype(str) + '_to_' + existing_storage_units.retirement_year.astype(str)
existing_storage_units.head()

c:\Users\ndematos\envs\pypsa_canada_py312\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


,bus,model,retirement_year,build_year,max_hours,p_nom,name
0,AB_Central,default_liion_battery,2045,2020,1.75,20,AB_Central_default_liion_battery_1.75h_2020_to...
1,AB_Central,default_liion_battery,2050,2025,1.75,60,AB_Central_default_liion_battery_1.75h_2025_to...
2,AB_North,default_liion_battery,2045,2020,1.75,20,AB_North_default_liion_battery_1.75h_2020_to_2045
3,AB_North,default_liion_battery,2050,2025,1.75,80,AB_North_default_liion_battery_1.75h_2025_to_2050
4,AB_South,default_liion_battery,2045,2020,2.00,10,AB_South_default_liion_battery_2.0h_2020_to_2045


In [65]:
planned_storage = pd.read_csv(os.path.join(path, 'data', 'planned_projects', 'new_storage.csv'))
planned_storage['geometry'] = planned_storage.apply(lambda row: Point(row.longitude, row.latitude), axis=1)
planned_storage = gpd.GeoDataFrame(planned_storage, geometry=planned_storage.geometry, index=planned_storage.index, crs='EPSG:4326').to_crs('EPSG: 3347')
planned_storage = planned_storage.sjoin_nearest(clusters, how='left')
planned_storage = planned_storage.drop(['geometry', 'province'], axis=1)
planned_storage = planned_storage.rename({'cluster':'bus'}, axis=1)

planned_storage['lifetime'] = 25
planned_storage['build_year'] = (5 * np.ceil((planned_storage['build_year']) / 5)).astype(int)
planned_storage['retirement_year'] = (5 * np.ceil((planned_storage['build_year'] + planned_storage['lifetime']) / 5)).astype(int)

planned_storage['lifetime'] = planned_storage['retirement_year'].subtract(planned_storage['build_year'])
planned_storage['name'] = planned_storage.name + '_' + planned_storage.model + '_' + planned_storage.build_year.astype(str) + '_to_' + planned_storage.retirement_year.astype(str)

planned_storage = planned_storage.drop(['latitude', 'longitude', 'lifetime'], axis=1)

existing_storage_units = pd.concat([existing_storage_units, planned_storage])
existing_storage_units.tail()

,bus,model,retirement_year,build_year,max_hours,p_nom,name
14,AB_Central,default_liion_battery,2055,2030,4.0,100.0,P2553 Dolcy MPC Solar and Storage_default_liio...
15,AB_Central,default_liion_battery,2055,2030,4.0,10.0,P2747 Vegreville 709S DER Battery_default_liio...
16,AB_South,default_liion_battery,2055,2030,4.0,10.1,P2924 Three Hills 770S DER Battery_default_lii...
17,AB_South,default_liion_battery,2055,2030,4.0,11.0,P2925 Michichi Creek 802S DER Battery_default_...
18,BC_South,hydro_monthly,2060,2035,744.0,500.0,Revelstoke Unit 6_hydro_monthly_2035_to_2060


In [66]:
# Non-extendable for existing storage units
existing_storage_units['p_nom_extendable'] = False
existing_storage_units['state_of_charge_initial'] = 0
existing_storage_units['state_of_charge_initial_per_period'] = False
existing_storage_units['cyclic_state_of_charge_per_period'] = False
existing_storage_units['committable'] = False

# Format storage units
existing_storage_units['lifetime'] = 25
existing_storage_units = existing_storage_units.set_index('name',drop=True)
existing_storage_units = existing_storage_units[['bus', 'model', 'p_nom', 'max_hours', 'state_of_charge_initial', 'state_of_charge_initial_per_period', 'cyclic_state_of_charge_per_period', 'p_nom_extendable', 'build_year', 'lifetime']]

existing_storage_units.head()

,bus,model,p_nom,max_hours,state_of_charge_initial,state_of_charge_initial_per_period,cyclic_state_of_charge_per_period,p_nom_extendable,build_year,lifetime
name,,,,,,,,,,
AB_Central_default_liion_battery_1.75h_2020_to_2045,AB_Central,default_liion_battery,20.0,1.75,0,False,False,False,2020,25
AB_Central_default_liion_battery_1.75h_2025_to_2050,AB_Central,default_liion_battery,60.0,1.75,0,False,False,False,2025,25
AB_North_default_liion_battery_1.75h_2020_to_2045,AB_North,default_liion_battery,20.0,1.75,0,False,False,False,2020,25
AB_North_default_liion_battery_1.75h_2025_to_2050,AB_North,default_liion_battery,80.0,1.75,0,False,False,False,2025,25
AB_South_default_liion_battery_2.0h_2020_to_2045,AB_South,default_liion_battery,10.0,2.00,0,False,False,False,2020,25


### Optional storage units


In [67]:
no_batteries = []
optional_storage = {}
for bus in buses.index:
    if bus in no_batteries:
        continue
    # Add optionnal lithium batteries
    optional_storage[f'{bus}_OPT_Battery'] = {
        'bus':bus,
        'model':'default_liion_battery',
        'p_nom':0,
        'p_nom_extendable':True,
        'max_hours':4,
        'state_of_charge_initial_per_period':True,
        'state_of_charge_initial': 0,
        'cyclic_state_of_charge_per_period':False,
        'build_year':BUILD_YEAR,
    }

optional_storage = pd.DataFrame.from_dict(optional_storage).T
optional_storage

,bus,model,p_nom,p_nom_extendable,max_hours,state_of_charge_initial_per_period,state_of_charge_initial,cyclic_state_of_charge_per_period,build_year
AB_Central_OPT_Battery,AB_Central,default_liion_battery,0,True,4,True,0,False,2030
AB_North_OPT_Battery,AB_North,default_liion_battery,0,True,4,True,0,False,2030
AB_South_OPT_Battery,AB_South,default_liion_battery,0,True,4,True,0,False,2030
BC_North_OPT_Battery,BC_North,default_liion_battery,0,True,4,True,0,False,2030
BC_South_OPT_Battery,BC_South,default_liion_battery,0,True,4,True,0,False,2030
MB_OPT_Battery,MB,default_liion_battery,0,True,4,True,0,False,2030
NB_East_OPT_Battery,NB_East,default_liion_battery,0,True,4,True,0,False,2030
NB_West_OPT_Battery,NB_West,default_liion_battery,0,True,4,True,0,False,2030
NL_Labrador_OPT_Battery,NL_Labrador,default_liion_battery,0,True,4,True,0,False,2030
NL_Newfoundland_OPT_Battery,NL_Newfoundland,default_liion_battery,0,True,4,True,0,False,2030


### Saving hydro storage unit data

In [68]:
storage_units = pd.concat([
    pd.read_csv(os.path.join(results_path, 'reservoir_storage_units.csv'), index_col=0),
    existing_storage_units,
    optional_storage])
storage_units.index.name = 'name'
storage_units.to_csv(os.path.join(model_results_path, 'storage_units.csv'))

inflows = pd.read_csv(os.path.join(results_path, 'storage_units-inflow.csv'), index_col=0).reset_index(drop=True)
inflows.index = pd.date_range(f'{REF_YEAR}-01-01', periods=8760, freq='h')
inflows.to_csv(os.path.join(model_results_path, 'storage_units-inflow.csv'))

## Load data

In [69]:
load_forecast = pd.read_csv(os.path.join(results_path, f'loads_forecast_{LOAD_SCENARIO}.csv'), index_col=0)
load_forecast.index = pd.to_datetime(load_forecast.index)
load_forecast.to_csv(os.path.join(model_results_path, f'loads_forecast_{LOAD_SCENARIO}.csv'))

loads = pd.read_csv(os.path.join(results_path, 'loads.csv'), index_col=0)
loads.to_csv(os.path.join(model_results_path, 'loads.csv'))

## Transmission interfaces

In [70]:
# Read results from line_distances
interface_data = pd.read_csv(os.path.join(results_path, 'cluster_interfaces.csv'))
interface_data = interface_data.rename(columns={'start':'bus0', 'end':'bus1', 'distance':'length', 'capacity_fwd':'p_nom'})

fwd_links = interface_data.copy()
fwd_links['name'] = fwd_links.bus0 + '->' + fwd_links.bus1
fwd_links['p_min_pu'] = 0

bck_links = interface_data[interface_data['capacity_bck'] > 0].copy()
# Flip the buses
bck_links[['bus0', 'bus1']] = bck_links[['bus1', 'bus0']].values
bck_links['p_nom'] = bck_links['capacity_bck']
bck_links['name'] = bck_links.bus0 + '->' + bck_links.bus1
bck_links['p_min_pu'] = 0

interface_data = pd.concat([fwd_links, bck_links], ignore_index=True)
interface_data = interface_data.set_index('name', drop=True)

interface_data['efficiency'] = 0.95
interface_data['build_year'] = 1
interface_data['lifetime'] = 'inf'
interface_data['p_nom_extendable'] = False
interface_data['capital_cost'] = 0
interface_data['carrier'] = 'DC'
interface_data['committable'] = False
interface_data = interface_data.drop(['dijkstra_distance', 'straight_distance', 'capacity_bck'], axis=1)
interface_data.head()

,bus0,bus1,length,p_nom,p_min_pu,efficiency,build_year,lifetime,p_nom_extendable,capital_cost,carrier,committable
name,,,,,,,,,,,,
AB_Central->AB_North,AB_Central,AB_North,437.000000,3588,0,0.95,1,inf,False,0,DC,False
AB_Central->AB_South,AB_Central,AB_South,281.850000,4565,0,0.95,1,inf,False,0,DC,False
AB_South->BC_South,AB_South,BC_South,683.598735,1000,0,0.95,1,inf,False,0,DC,False
AB_South->SK,AB_South,SK,648.898048,150,0,0.95,1,inf,False,0,DC,False
BC_North->BC_South,BC_North,BC_South,634.665000,11546,0,0.95,1,inf,False,0,DC,False


### Expansion options

In [71]:
tx_price = 85.11 # $CAD/km/MW, annualized
optional_links = {}
for name, data in interface_data.iterrows():
    optional_links[f'OPT_Link_{data.bus0}_to_{data.bus1}'] = {
        'bus0':data.bus0,
        'bus1':data.bus1,
        'length':data.length,
        'p_nom':0,
        'p_min_pu':0,
        'efficiency':0.95,
        'build_year':BUILD_YEAR,
        'lifetime':40,
        'p_nom_extendable':True,
        'carrier':'DC',
        'committable':False
    }
    optional_links[f'OPT_Link_{data.bus0}_to_{data.bus1}']['capital_cost'] = round(tx_price/2 * data.length, 2)
optional_links = pd.DataFrame.from_dict(optional_links).T
optional_links

,bus0,bus1,length,p_nom,p_min_pu,efficiency,build_year,lifetime,p_nom_extendable,carrier,committable,capital_cost
OPT_Link_AB_Central_to_AB_North,AB_Central,AB_North,437.0,0,0,0.95,2030,40,True,DC,False,18596.53
OPT_Link_AB_Central_to_AB_South,AB_Central,AB_South,281.85,0,0,0.95,2030,40,True,DC,False,11994.13
OPT_Link_AB_South_to_BC_South,AB_South,BC_South,683.598735,0,0,0.95,2030,40,True,DC,False,29090.54
OPT_Link_AB_South_to_SK,AB_South,SK,648.898048,0,0,0.95,2030,40,True,DC,False,27613.86
OPT_Link_BC_North_to_BC_South,BC_North,BC_South,634.665,0,0,0.95,2030,40,True,DC,False,27008.17
OPT_Link_MB_to_ON_Northwest,MB,ON_Northwest,622.378154,0,0,0.95,2030,40,True,DC,False,26485.3
OPT_Link_MB_to_SK,MB,SK,512.161629,0,0,0.95,2030,40,True,DC,False,21795.04
OPT_Link_NB_East_to_NB_West,NB_East,NB_West,211.24,0,0,0.95,2030,40,True,DC,False,8989.32
OPT_Link_NB_East_to_NS_West,NB_East,NS_West,164.280937,0,0,0.95,2030,40,True,DC,False,6990.98
OPT_Link_NB_East_to_PE,NB_East,PE,73.687412,0,0,0.95,2030,40,True,DC,False,3135.77


In [72]:
interface_data = pd.concat([
    interface_data,
    optional_links
])
interface_data.index.name = 'name'
interface_data.to_csv(os.path.join(model_results_path, 'links.csv'))

# Snapshots File
Needed to load model

In [73]:
snapshots = pd.DataFrame(index=load_forecast.loc[str(REF_YEAR)].index)
snapshots['weightings'] = 1
snapshots.to_csv(os.path.join(model_results_path, 'snapshots.csv'))

## Validation

In [74]:
network = pypsa.Network(model_results_path)
network

INFO:pypsa.network.io:New version 1.2.4 available! (Current: 1.2.1)
INFO:pypsa.network.index:Applying weightings to all columns of `snapshot_weightings`
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, generators, links, loads, storage_units


PyPSA Network 'Unnamed Network'
-------------------------------
Components:
 - Bus: 23
 - Generator: 493
 - Link: 106
 - Load: 33
 - StorageUnit: 87
Snapshots: 8760

In [26]:
#Due to incompatibilities with linopy and xarray, pyarrow will be removed here
%pip uninstall -y pyarrow

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
